# Text classification with RNNs

In this lab, we will find the author of a sentence with recurrent networks. The corpus is the slides' one: the sentences of 2 works by each of 9 English-language authors, from 1821 to 1913 (Brontë, Melville, Wilde…).

1. From text to sequences of indices, with `TextVectorization`.
2. A recurrent classifier that ignores the padding.
3. Training a bidirectional GRU.
4. Without a mask, the GRU reads the padding.
5. Simple or bidirectional GRU?
6. Pretrained, frozen GloVe embeddings.
7. A deeper model.
8. The authors the model mixes up.

This lab needs a GPU: check in `Runtime > Change runtime type` that the hardware accelerator is a GPU. Each training takes from 10 seconds to a minute and a quarter (the deep model of part 7) on an RTX 3090; Colab's T4 is about two to three times slower. On a CPU, it would take from a quarter of an hour to several hours.

## Downloads

The corpus, spaCy's English model and the GloVe vectors of part 6 (an 822 MB file): allow less than a minute.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh classical-authors
!python -m spacy download en_core_web_sm
!wget -q https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip
!unzip -q -o glove.6B.zip glove.6B.300d.txt

## Importing the libraries

In [ ]:
import pathlib
from collections.abc import Iterable, Iterator

import keras
import matplotlib.pyplot as plt
import numpy as np
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import spacy
import tqdm.auto
from keras import layers

# Same random draws at each run: reproducible results
keras.utils.set_random_seed(0)

## Preprocessing

The next cell prepares the data, as in the slides:

- each sentence, split by spaCy, is an example, and its label is its author, from 0 to 8;
- every named entity (a word or a group of words) is replaced with its type, first letter tripled to make it a new word (`PPPERSON`, `DDDATE`…): the model must recognize a style, not the characters' names;
- the corpus' directories have misspelled names (`Melwille`, `bronteCharlote`): `AUTHORS` gives each author's name;
- a sentence without any word (an opening bracket "[" split off by spaCy, punctuation alone) is skipped: empty once vectorized, these 1,763 sentences would be fully masked, which cuDNN, the kernel of Keras's recurrent layers on GPU, rejects;
- 70% of the sentences are for training (`X_train_raw`, `y_train`), 30% for testing (`X_test_raw`, `y_test`).

It takes about a minute.

In [ ]:
# Corpus directory → author's name
AUTHORS = {
  "bronteAnne": "Anne Brontë",
  "bronteCharlote": "Charlotte Brontë",
  "Cooper": "Cooper",
  "Emerson": "Emerson",
  "Hawthorne": "Hawthorne",
  "Melwille": "Melville",
  "Shaw": "Shaw",
  "Thoreau": "Thoreau",
  "wilde": "Wilde",
}

nlp = spacy.load(
  "en_core_web_sm",
  exclude=("tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"),
)
nlp.enable_pipe("senter")


def replace_ners(tokens: Iterable[spacy.tokens.Token]) -> Iterator[str]:
  for token in tokens:
    if token.ent_iob_ == "O":
      yield token.text
    elif token.ent_iob_ == "B":
      prefix = token.ent_type_[0]
      yield f"{prefix}{prefix}{token.ent_type_}"


def get_data(directory: pathlib.Path) -> tuple[list[str], list[str]]:
  texts = []
  authors = []
  # <author>/<work>/<chapter>.txt
  paths = sorted(directory.glob("*/*/*.txt"))
  contents = (p.read_text(encoding="utf8").replace("\n", " ") for p in paths)
  for path, doc in tqdm.auto.tqdm(zip(paths, nlp.pipe(contents)), total=len(paths)):
    author = AUTHORS[path.parent.parent.name]
    for sentence in doc.sents:
      text = " ".join(replace_ners(sentence))
      # A sentence without any letter or digit would be empty once vectorized
      if any(char.isalnum() for char in text):
        authors.append(author)
        texts.append(text)
  return texts, authors


texts, authors = get_data(pathlib.Path("classical-authors"))
label_encoder = sklearn.preprocessing.LabelEncoder()
y = label_encoder.fit_transform(authors)
print(f"{len(texts)} sentences; labels 0 to 8: {', '.join(label_encoder.classes_)}")

X_train_raw, X_test_raw, y_train, y_test = sklearn.model_selection.train_test_split(
  texts, y, test_size=0.3, random_state=0
)
print(f"Training: {len(X_train_raw)} sentences, test: {len(X_test_raw)}")
for text, label in zip(X_train_raw[:3], y_train[:3]):
  print(f"{label_encoder.classes_[label]}: {text}")

## 1. From text to sequences of indices

The next cell turns the sentences into tensors of word indices, as in the slides, with a `layers.TextVectorization` layer, `vectorizer`: a vocabulary of the 20,000 most frequent words, learned from the training sentences only and stored in `vocabulary`, then 150 indices per sentence, a shorter sentence being completed with 0s at its end (the padding), a longer one truncated.

In [ ]:
vectorizer = layers.TextVectorization(
  max_tokens=20_000,  # the 20,000 most frequent words
  output_sequence_length=150,  # 0s added at the end, or sentence truncated
)
vectorizer.adapt(X_train_raw)  # vocabulary of the training set only
X_train = vectorizer(X_train_raw)
X_test = vectorizer(X_test_raw)
vocabulary = vectorizer.get_vocabulary()

print(X_train.shape)
print(X_train_raw[0])
print(X_train[0, :20].numpy())
print(" | ".join(vocabulary[:10]))

### How many words to keep?

The two choices of the previous cell, 150 indices per sentence and a vocabulary of 20,000 words, are decided from the data. Measure them:

- `lengths`, the number of words of each training sentence: the nonzero indices of its row of `X_train` (a sentence longer than 150 words counts as 150 there);
- the histogram of `lengths`, its median and its 99th percentile;
- the share of the training sentences that are truncated, those filling their 150 indices;
- the share of the words of the test sentences out of the vocabulary (index 1), and the one of the test sentences with at least one of them.

Which maximum length would you choose from this distribution? What do 150 indices lose, and what do they cost? Are the vocabulary's 20,000 words enough?

**Useful documentation**:

- https://numpy.org/doc/stable/reference/generated/numpy.count_nonzero.html
- https://numpy.org/doc/stable/reference/generated/numpy.percentile.html
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html

In [ ]:
# Your code here

#### Additional hints

The calls and the parameters to use:

- `np.count_nonzero(X_train, axis=1)`: the number of nonzero indices of each row
- `plt.hist(lengths, bins=75)`: 75 bars 2 words wide; `plt.yscale("log")` makes the long, rare sentences visible
- `np.median(lengths)`, `np.percentile(lengths, 99)`: half of the sentences are shorter than the median, 99 out of 100 than the 99th percentile
- `(lengths == 150).mean()`: the share of truncated sentences
- `test = X_test.numpy()`, then `(test == 1).sum() / (test != 0).sum()`: the share of unknown words; `(test == 1).any(axis=1)` tells for each sentence whether it has at least one

A skeleton to copy into the exercise cell, replacing each `...`:

```python
lengths = np.count_nonzero(..., axis=1)
plt.hist(lengths, bins=...)
plt.yscale("log")
plt.xlabel("Words per sentence")
plt.ylabel("Sentences")
plt.show()
print(f"Median: {np.median(...):.0f} words; 99th percentile: {np.percentile(...):.0f}")
print(f"Truncated sentences: {(lengths == ...).mean():.1%}")
test = X_test.numpy()
print(f"Unknown words in test: {(test == ...).sum() / (test != ...).sum():.1%}")
print(f"Sentences with an unknown word: {(test == ...).any(axis=1).mean():.0%}")
```

#### Solution

In [ ]:
lengths = np.count_nonzero(X_train, axis=1)  # words of each sentence, at most 150
plt.hist(lengths, bins=75)
plt.yscale("log")  # long sentences are rare
plt.xlabel("Words per sentence")
plt.ylabel("Sentences")
plt.show()
print(
  f"Median: {np.median(lengths):.0f} words; "
  f"99th percentile: {np.percentile(lengths, 99):.0f}"
)
print(f"Truncated sentences: {(lengths == 150).mean():.1%}")
test = X_test.numpy()
print(f"Unknown words in test: {(test == 1).sum() / (test != 0).sum():.1%}")
print(f"Sentences with an unknown word: {(test == 1).any(axis=1).mean():.0%}")

# A sentence has 17 words at the median, and 99% have fewer than 100: the 99th
# percentile, 100 indices, would do. 150 truncate only 0.2%; the 50 extra indices cost
# no accuracy, since the mask makes the GRU skip the padding (part 2), but a little
# computation: the GRU goes through the 150 steps of each batch. The 20,000 words kept,
# out of the training sentences' 32,603, cover 98% of the test words: 1.9% are unknown
# (rare words, names spaCy didn't recognize). But 27% of the test sentences have at
# least one: a larger vocabulary would save a few, at the cost of a larger embedding
# table, for words too rare to be learned well in it.

## 2. A recurrent classifier that ignores the padding

The function `build_model` of the next cell is the slides' one, with one more parameter, `embedding`: the embedding layer, a new 64-dimensional `layers.Embedding` layer if none is given (the GloVe vectors of part 6). The next parts call it with the recurrent layer(s) to compare, for instance `build_model(layers.GRU(64))`.

The mask is essential here: a sentence has 17 words at the median, followed by more than 130 padding zeros. `mask_zero=True` computes the mask of the 0 indices, which the embedding passes on to the recurrent layers: they skip these steps, and their final state is the one of the last real word.

In [ ]:
def build_model(
  *recurrent_layers: layers.Layer, embedding: layers.Embedding | None = None
) -> keras.Sequential:
  if embedding is None:
    # A vector of 64 numbers per word, and the mask of the 0 indices
    embedding = layers.Embedding(20_000, 64, mask_zero=True)
  return keras.Sequential(
    [
      keras.Input(shape=(None,), dtype="int32"),  # sentences of any length
      embedding,
      *recurrent_layers,
      layers.Dense(len(label_encoder.classes_), activation="softmax"),
    ]
  )

## 3. Training a bidirectional GRU

The function `train` compiles a model, trains it and evaluates it as in the slides:

- loss `sparse_categorical_crossentropy`, since the labels are integers; Adam optimizer, with clipping of the gradient's norm (`clipnorm`);
- batches of 256 sentences; 20% of the training sentences for validation;
- at most 20 epochs: `EarlyStopping` stops the training after 2 epochs without a decrease of the validation loss, and puts back the weights of the best one.

It plots the accuracy curves and returns the test accuracy. The next cell applies it to a bidirectional GRU with 64 units per reading direction.

In [ ]:
def train(model: keras.Model, epochs: int = 20) -> float:
  model.compile(
    optimizer=keras.optimizers.Adam(clipnorm=1.0),  # gradient clipping
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  stop = keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)
  history = model.fit(
    X_train,
    y_train,
    batch_size=256,
    epochs=epochs,
    validation_split=0.2,
    callbacks=[stop],
  )
  _, accuracy = model.evaluate(X_test, y_test, verbose=0)

  # Accuracies of the best epoch, the one whose weights are kept
  best = int(np.argmin(history.history["val_loss"]))
  print(
    f"Best epoch: {best + 1}; training accuracy "
    f"{history.history['accuracy'][best]:.3f}, "
    f"validation {history.history['val_accuracy'][best]:.3f}, test {accuracy:.3f}"
  )
  epochs_run = np.arange(1, len(history.epoch) + 1)
  plt.plot(epochs_run, history.history["accuracy"], label="training")
  plt.plot(epochs_run, history.history["val_accuracy"], label="validation")
  plt.axvline(best + 1, color="gray", linestyle="--", label="best epoch")
  plt.xlabel("Epoch")
  plt.ylabel("Accuracy")
  plt.legend()
  plt.show()
  return accuracy

In [ ]:
bigru = build_model(layers.Bidirectional(layers.GRU(64)))
bigru.summary()
bigru_accuracy = train(bigru)

### First conclusions

Given the curves and the accuracies, what can be said of the quality of this training? Compare the test accuracy with chance (one in 9) and with a model that would always predict the most frequent author, Melville (18% of the sentences).

Your answer here

#### Solution

The model learns: about 55% test accuracy, five times chance (11%) and three times the majority class (18%).

But it overfits from the 3rd epoch: its training accuracy keeps rising (75% at the 4th epoch), while the validation one levels off around 57% and the validation loss goes back up. `EarlyStopping` stops the training two epochs later and puts back the weights of the best epoch.

The model has 1.3 million parameters, 1.28 million of them in the embedding table, for 41,000 training sentences: it memorizes the words of the training sentences rather than each author's style.

## 4. Without a mask, the GRU reads the padding

What does the mask bring? Train with `train` the same bidirectional GRU as in part 3, but on a `layers.Embedding` layer without a mask: index 0 becomes a word like any other there, which the GRU reads. Compare their test accuracies.

In `X_train`, a 17-word sentence is followed by 133 zeros. What does each reading direction of the GRU end its reading with? Which of the two does the padding hinder?

In [ ]:
# Your code here

### Additional hints

The calls and the parameters to use:

- `layers.Embedding(20_000, 64)`: without `mask_zero=True`, no mask is computed, nor passed on to the GRU
- `build_model` takes the embedding layer through its `embedding` parameter, after the recurrent layers
- `bigru_accuracy` holds the test accuracy of the bidirectional GRU of part 3

A skeleton to copy into the exercise cell, replacing each `...`:

```python
unmasked_bigru = build_model(
  layers.Bidirectional(...), embedding=layers.Embedding(..., ...)
)
unmasked_accuracy = train(...)
print(f"With a mask: {bigru_accuracy:.3f}, padding read: {...:.3f}")
```

### Solution

In [ ]:
unmasked_bigru = build_model(
  layers.Bidirectional(layers.GRU(64)),
  embedding=layers.Embedding(20_000, 64),  # without mask_zero: the padding is read
)
unmasked_accuracy = train(unmasked_bigru)
print(f"With a mask: {bigru_accuracy:.3f}, padding read: {unmasked_accuracy:.3f}")

Without a mask, the bidirectional GRU does a little worse: about 53% test accuracy, against 55% with the mask.

Its forward direction reads the sentence, then its 133 zeros: its final state is the one of the last zero, and what it has kept of the sentence must go through more than 130 steps without information. Its backward direction starts with the zeros, then reads the sentence backward: its final state is the one of the first word, and it still summarizes the whole sentence. So the padding mostly hinders the forward direction, and the model hardly relies on anything but the backward one: its accuracy is about that of the GRU with a single reading direction of part 5. The gap stays modest, because the backward direction almost suffices; a simple GRU would only have its forward direction, and the mask would matter much more there.

## 5. Simple or bidirectional GRU?

Train with `train` a simple GRU with 64 units, which reads the sentences in one direction only, and compare it with the bidirectional GRU of part 3:

- the test accuracy;
- the number of parameters of the recurrent layer, and of the whole model;
- the duration of an epoch, which Keras shows during the training.

Is the second reading direction worth its cost here?

In [ ]:
# Your code here

### Additional hints

- `build_model(layers.GRU(64))`: the same model, with a simple GRU
- `model.summary()` gives the number of parameters of each layer; `model.count_params()` the one of the whole model
- `bigru_accuracy` holds the test accuracy of the bidirectional GRU

A skeleton to copy into the exercise cell, replacing each `...`:

```python
gru = build_model(...)
gru.summary()
gru_accuracy = train(...)
print(f"GRU: {gru_accuracy:.3f}, {gru.count_params()} parameters")
print(f"Bidirectional GRU: {...:.3f}, {bigru.count_params()} parameters")
```

### Solution

In [ ]:
gru = build_model(layers.GRU(64))
gru.summary()
gru_accuracy = train(gru)
print(f"GRU: {gru_accuracy:.3f}, {gru.count_params()} parameters")
print(f"Bidirectional GRU: {bigru_accuracy:.3f}, {bigru.count_params()} parameters")

The simple GRU does a little worse: about 53% test accuracy, against 54 to 56% for the bidirectional GRU, depending on the run.

The second reading direction doubles the parameters of the recurrent layer (24,960 → 49,920), but that is only 2% of the model, mostly made of the embedding (1.28 million); an epoch lasts about 40% longer (11 ms per batch against 8 on an RTX 3090). The gain is modest, but worth this cost: the final state of a simple GRU mostly summarizes the end of the sentence, and the backward direction adds a summary of its beginning.

## 6. Pretrained, frozen GloVe embeddings

The embeddings of the previous models start from random vectors and are learned on our 41,000 training sentences. The [GloVe](https://nlp.stanford.edu/projects/glove/) vectors were learned on 6 billion words (Wikipedia and Gigaword): words with close meanings have close vectors there.

Read the file `glove.6B.300d.txt` into a dictionary `glove`, word → vector: each line is a word followed by its 300 coordinates, separated by spaces, and only the words of `vocabulary` matter to us. The cell after the solution turns it into an embedding layer, as in the slide "Pretrained Embeddings: GloVe".

How many of the vocabulary's 20,000 words does GloVe know? Which ones are missing, and why?

In [ ]:
# Your code here

### Additional hints

The calls and the parameters to use:

- `word, *coordinates = line.rstrip().split(" ")` separates the word from its coordinates; `np.array(coordinates, dtype="float32")` makes them a vector
- `set(vocabulary)`: a fast `word in ...` test over the file's 400,000 lines

A skeleton to copy into the exercise cell, replacing each `...`:

```python
words = set(vocabulary)
glove = {}
with open("glove.6B.300d.txt", encoding="utf8") as file:
  for line in file:
    word, *coordinates = line.rstrip().split(" ")
    if word in ...:
      glove[word] = np.array(coordinates, dtype="float32")

missing = [word for word in vocabulary if word not in ...]
print(f"{len(glove)} words out of {len(vocabulary)} in GloVe")
print("Missing:", ", ".join(missing[:30]))
```

### Solution

In [ ]:
# Only the vectors of the vocabulary's words are kept
words = set(vocabulary)
glove = {}
with open("glove.6B.300d.txt", encoding="utf8") as file:
  for line in file:
    word, *coordinates = line.rstrip().split(" ")
    if word in words:
      glove[word] = np.array(coordinates, dtype="float32")

missing = [word for word in vocabulary if word not in glove]
print(f"{len(glove)} words out of {len(vocabulary)} in GloVe")
print("Missing:", ", ".join(missing[:30]))

# 19,102 of the vocabulary's 20,000 words are in GloVe. Missing: the padding and [UNK]
# (indices 0 and 1), the entity types (ppperson, dddate…), names spaCy didn't recognize
# (holgrave, pyncheon), the vocabulary of whaling (harpooneer, whalemen) and old
# spellings (oftener, trowsers, noontide): words that a 21st-century corpus (Wikipedia,
# the press) doesn't use.

### The GloVe embedding layer

The next cell builds `glove_embedding` as in the slide "Pretrained Embeddings: GloVe":

- `embedding_matrix`, of shape `(len(vocabulary), 300)`: its row `i` is the vector of the word `vocabulary[i]`, or stays zero if GloVe doesn't know this word;
- a `layers.Embedding` layer initialized with this matrix (`embeddings_initializer`), frozen (`trainable=False`: its vectors don't change during training), and which computes the padding's mask (`mask_zero=True`).

In [ ]:
embedding_matrix = np.zeros((len(vocabulary), 300))  # 0 if the word is missing
for i, word in enumerate(vocabulary):
  if word in glove:
    embedding_matrix[i] = glove[word]
glove_embedding = layers.Embedding(
  input_dim=len(vocabulary),
  output_dim=300,
  mask_zero=True,
  embeddings_initializer=keras.initializers.Constant(embedding_matrix),
  trainable=False,  # frozen table
)

### Training with GloVe

Train the same bidirectional GRU as in part 3, with `glove_embedding` as its embedding layer, and compare it with the first one:

- the gap between the training accuracy and the validation one;
- the test accuracy;
- the number of trainable parameters, which `summary()` gives.

In [ ]:
# Your code here

#### Additional hints

`build_model` takes the embedding layer through its `embedding` parameter, after the recurrent layers. `bigru_accuracy` holds the test accuracy of the first model.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
glove_bigru = build_model(layers.Bidirectional(...), embedding=...)
glove_bigru.summary()
glove_accuracy = train(...)
print(f"Learned embeddings: {bigru_accuracy:.3f}, frozen GloVe: {glove_accuracy:.3f}")
```

#### Solution

In [ ]:
glove_bigru = build_model(
  layers.Bidirectional(layers.GRU(64)), embedding=glove_embedding
)
glove_bigru.summary()
glove_accuracy = train(glove_bigru)
print(f"Learned embeddings: {bigru_accuracy:.3f}, frozen GloVe: {glove_accuracy:.3f}")

With frozen GloVe, the model has only 141,705 trainable parameters, against 1.3 million: the 6 million numbers of the GloVe table don't move.

It overfits much more slowly: its validation loss decreases for 8 epochs instead of 2, down to a lower minimum, and the test accuracy improves a little, from about 55% to 57%. The GloVe vectors bring what our 41,000 sentences can't learn alone: words with close meanings have close vectors there, even when they are rare in the corpus.

Yet the gap between the training accuracy and the validation one ends up widening: the recurrent layer still memorizes. This is the role of the next model's dropout.

## 7. A deeper model

A last model, given: two bidirectional GRUs stacked on the GloVe embeddings.

- `return_sequences=True`: the first layer returns one state per word, the sequence the second one reads;
- `dropout=0.5`: half of the input values of each recurrent layer set to zero at each training step, in both reading directions, against overfitting.

In [ ]:
deep = build_model(
  layers.Bidirectional(layers.GRU(64, dropout=0.5, return_sequences=True)),
  layers.Bidirectional(layers.GRU(64, dropout=0.5)),
  embedding=glove_embedding,
)
deep.summary()
deep_accuracy = train(deep)
print(
  f"GloVe and one bidirectional GRU: {glove_accuracy:.3f}, two: {deep_accuracy:.3f}"
)

The deep model reaches about 58% test accuracy, barely more than the bidirectional GRU on GloVe. Dropout has removed the overfitting: the training and validation accuracies stay equal, around 59% at the end. But it slows the learning down: the validation loss still decreases at the 20th epoch, and `EarlyStopping` didn't have to stop the training. A few more epochs would probably still gain a little.

## 8. The authors the model mixes up

The next cell shows the deep model's confusion matrix on the test sentences: an author's row gives how their sentences are spread among the predicted authors.

In [ ]:
y_pred = deep.predict(X_test, batch_size=1024).argmax(axis=1)
fig, ax = plt.subplots(figsize=(8, 8))
sklearn.metrics.ConfusionMatrixDisplay.from_predictions(
  y_test,
  y_pred,
  display_labels=label_encoder.classes_,
  normalize="true",  # proportions of each row
  values_format=".0%",
  xticks_rotation=45,
  cmap="Blues",
  colorbar=False,
  ax=ax,
)
ax.set_xlabel("Predicted author")
ax.set_ylabel("True author")
plt.show()

### Reading the matrix

Which authors does the model recognize best? Which ones does it mix up, and what do they have in common? The corpus' works are listed in the [dataset's README](https://github.com/shuuchuu/datasets/tree/main/classical-authors).

Your answer here

#### Solution

The diagonal gives the share of each author's sentences rightly attributed; these figures vary by a few points from one run to another.

- Wilde (68%), Melville (65%) and Thoreau (64%) are the best recognized. *Moby-Dick* has a vocabulary of its own (the sea, whaling), so do Thoreau's two books (nature, the river, the woods), and Wilde's plays are made of short lines, in a spoken style, which look neither like the novels nor like the essays.
- Anne Brontë is the least recognized (43%): 24% of her sentences are attributed to her sister Charlotte. They are two English novelists of the same period, whose novels are told in the first person, by a governess in *Agnes Grey* as in *Jane Eyre*. The confusion mostly goes this way (only 7% of Charlotte's sentences go to Anne): Charlotte, almost twice as represented, attracts the ambiguous sentences, and not only Anne's (from 4 to 10% of the sentences of each of the other authors).
- Shaw and Wilde get mixed up: 18% of Shaw's sentences are taken for Wilde, 13% of Wilde's for Shaw. They are the corpus' two Irish playwrights, and their lines look alike.
- So do Emerson and Thoreau: 22% of Emerson's sentences are attributed to Thoreau, 12% the other way round. They are two American essayists of transcendentalism, neighbors and friends.

A single sentence is weak evidence: many are short and commonplace ("Yes ."), and no model can attribute them. To put these results in perspective, a logistic regression on a TF-IDF representation of the sentences reaches about 61% accuracy on the same split: on short sentences, the words used matter more than their order.